### **MNIST**

In [1]:
import numpy as np
from src.gmm import GMM
from src.knn import KNN


### **Load Data**

In [2]:
import os
from PIL import Image


def load_images(folder):
    images = []
    labels = []
    for digit in [2, 3, 9]:
        digit_folder = f"{folder}/{digit}"
        for filename in sorted(os.listdir(digit_folder)):
            if filename.endswith(".png"):
                with Image.open(f"{digit_folder}/{filename}") as image:
                    pixels = np.array(image.convert("L"), dtype=float)
                    images.append(pixels.ravel() / 255.0)
                    labels.append(digit)
    return np.array(images), np.array(labels)


X_train, y_train = load_images("./data/mnist/train")
X_val, y_val = load_images("./data/mnist/dev")
X_test, y_test = load_images("./data/mnist/test")


In [3]:
classes = np.unique(y_train)
print("Class order:", classes)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Class order: [2 3 9]
Train: (14424, 784)
Validation: (3614, 784)
Test: (3051, 784)


### **GMM**

In [4]:
models = []
priors = []
for label in classes:
    samples = X_train[y_train == label]
    model = GMM(K=3, reg=1e-3, max_iter=100, tol=1e-4)
    model.fit(samples)
    models.append(model)
    priors.append(len(samples) / len(X_train))
    print("Class", label, "EM iterations:", len(model.log_likelihoods))
    if len(model.log_likelihoods) == model.max_iter:
        print("Iteration limit reached; convergence is not guaranteed.")


Class 2 EM iterations: 17
Class 3 EM iterations: 25
Class 9 EM iterations: 38


In [5]:
val_scores = np.column_stack([model.score_samples(X_val) for model in models])
gmm_val_predictions = classes[np.argmax(val_scores + np.log(priors), axis=1)]

test_scores = np.column_stack([model.score_samples(X_test) for model in models])
gmm_test_predictions = classes[np.argmax(test_scores + np.log(priors), axis=1)]


In [6]:
# Confusion matrix
gmm_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_confusion)


Validation accuracy: 0.966519092418373
Test accuracy: 0.97312356604392
Test confusion matrix:
 [[1006   21    5]
 [  12  989    9]
 [   5   30  974]]


### **KNN**

In [7]:
knn = KNN(k=5)
knn.fit(X_train, y_train)
knn_val_predictions = knn.predict(X_val)
knn_test_predictions = knn.predict(X_test)


In [8]:
# Confusion matrix
knn_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_test_predictions == y_test))
print("Test confusion matrix:\n", knn_confusion)


Validation accuracy: 0.9858882125069175
Test accuracy: 0.9891838741396264
Test confusion matrix:
 [[1027    2    3]
 [   2  998   10]
 [   5   11  993]]
